# 06 - DistilBERT Full Training

This notebook just runs a cell to train DistilBERT over the full dataset for submission to HuggingFace. 

In [1]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Data — full dataset, no split ─────────────────────────────────────────────
df = pd.read_csv('../data/processed/combined_base_data.csv')
df = df.dropna(subset=['title']).reset_index(drop=True)

X = df['title'].tolist()
y = df['is_fox'].tolist()
print(f'Total: {len(X):,} | Fox%: {np.mean(y):.2%}')

# ── Dataset + DataLoader ───────────────────────────────────────────────────────
MAX_LENGTH = 64
BATCH_SIZE = 32
tokenizer  = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class HeadlineDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts, truncation=True, max_length=max_length,
            padding='max_length', return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx],
        }

train_loader = DataLoader(HeadlineDataset(X, y, tokenizer, MAX_LENGTH),
                          batch_size=BATCH_SIZE, shuffle=True)
print(f'Train batches: {len(train_loader)}')

# ── Model ──────────────────────────────────────────────────────────────────────
NUM_EPOCHS = 40 # Pick 40 because gains tended to plateau after 30s
LR = 2e-5
total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * 0.1)

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
).to(device)

for param in model.distilbert.parameters():
    param.requires_grad = False
    
for param in model.distilbert.transformer.layer[-1].parameters():
    param.requires_grad = True

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=0.01
)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}')

# ── Training loop — no validation, 40 epochs ────────────────────
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=-1)
        correct += (preds == batch['labels']).sum().item()
        total   += len(batch['labels'])

    train_acc = correct / total
    print(f'Epoch {epoch}/{NUM_EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Train acc: {train_acc:.4f}')

torch.save(model.state_dict(), '../model_full.pt')
print('Saved to ../model_full.pt')

/Users/rohankrishnan/Documents/GitHub/cis-5190-news/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
Total: 3,801 | Fox%: 52.62%
Train batches: 119


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11472.07it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 7,680,002
Total steps: 4760 | Warmup steps: 476
Epoch 1/40 | Loss: 0.6908 | Train acc: 0.5296
Epoch 2/40 | Loss: 0.6610 | Train acc: 0.6333
Epoch 3/40 | Loss: 0.5656 | Train acc: 0.7288
Epoch 4/40 | Loss: 0.4967 | Train acc: 0.7693
Epoch 5/40 | Loss: 0.4329 | Train acc: 0.8024
Epoch 6/40 | Loss: 0.3917 | Train acc: 0.8285
Epoch 7/40 | Loss: 0.3590 | Train acc: 0.8443
Epoch 8/40 | Loss: 0.3306 | Train acc: 0.8564
Epoch 9/40 | Loss: 0.2972 | Train acc: 0.8732
Epoch 10/40 | Loss: 0.2812 | Train acc: 0.8800
Epoch 11/40 | Loss: 0.2576 | Train acc: 0.8892
Epoch 12/40 | Loss: 0.2447 | Train acc: 0.9016
Epoch 13/40 | Loss: 0.2180 | Train acc: 0.9140
Epoch 14/40 | Loss: 0.2031 | Train acc: 0.9171
Epoch 15/40 | Loss: 0.1894 | Train acc: 0.9266
Epoch 16/40 | Loss: 0.1713 | Train acc: 0.9374
Epoch 17/40 | Loss: 0.1589 | Train acc: 0.9421
Epoch 18/40 | Loss: 0.1544 | Train acc: 0.9458
Epoch 19/40 | Loss: 0.1416 | Train acc: 0.9450
Epoch 20/40 | Loss: 0.1287 | Train acc: 0.9526

In [3]:
import torch
print(list(torch.load('../scripts/distilbert/model.pt', map_location='cpu').keys())[:10])

['distilbert.embeddings.word_embeddings.weight', 'distilbert.embeddings.position_embeddings.weight', 'distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias', 'distilbert.transformer.layer.0.attention.q_lin.weight', 'distilbert.transformer.layer.0.attention.q_lin.bias', 'distilbert.transformer.layer.0.attention.k_lin.weight', 'distilbert.transformer.layer.0.attention.k_lin.bias', 'distilbert.transformer.layer.0.attention.v_lin.weight', 'distilbert.transformer.layer.0.attention.v_lin.bias']


In [5]:
keys = list(torch.load('../scripts/distilbert/model.pt', map_location='cpu').keys())
print(f'Total keys: {len(keys)}')
print(keys[-10:])

Total keys: 104
['distilbert.transformer.layer.5.ffn.lin1.weight', 'distilbert.transformer.layer.5.ffn.lin1.bias', 'distilbert.transformer.layer.5.ffn.lin2.weight', 'distilbert.transformer.layer.5.ffn.lin2.bias', 'distilbert.transformer.layer.5.output_layer_norm.weight', 'distilbert.transformer.layer.5.output_layer_norm.bias', 'pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']


In [6]:
import torch
ckpt = torch.load('../scripts/distilbert/model.pt', map_location='cpu')
print([k for k in ckpt.keys() if 'classifier' in k])

['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
